# Spatial Structure: Distance, Weights, and Moran's I

This notebook moves from coordinates to a spatial weights matrix and then to a permutation test for global spatial autocorrelation.

The key lesson is that **the neighborhood definition is part of the analysis**. Moran's \(I\) has no meaning without the weights matrix \(W\) and the null reference used for inference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

rng = np.random.default_rng(7)
xx, yy = np.meshgrid(np.linspace(0, 1, 12), np.linspace(0, 1, 12))
coords = np.column_stack([xx.ravel(), yy.ravel()])
D = cdist(coords, coords)

covariance = np.exp(-D / 0.22) + 1e-10 * np.eye(len(coords))
values = rng.multivariate_normal(np.zeros(len(coords)), covariance)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=values, s=45)
fig.colorbar(sc, ax=ax, label="simulated value")
ax.set(title="Spatially correlated Gaussian field", xlabel="x", ylabel="y")
plt.show()


## Build a \(k\)-nearest-neighbor matrix

We construct binary \(k\)-nearest-neighbor weights, symmetrize them, and then row-standardize. Row standardization makes each non-island row sum to one, so \(Wz\) is a neighbor average of centered values.

In [ ]:
k = 4
D_for_knn = D.copy()
np.fill_diagonal(D_for_knn, np.inf)

W = np.zeros_like(D_for_knn)
nbr = np.argpartition(D_for_knn, kth=k-1, axis=1)[:, :k]
W[np.repeat(np.arange(len(coords)), k), nbr.ravel()] = 1
W = np.maximum(W, W.T)

row_sums = W.sum(axis=1)
W = W / row_sums[:, None]

print("neighbor counts before standardization:", np.unique(row_sums, return_counts=True))
print("row sums after standardization:", np.unique(np.round(W.sum(axis=1), 12)))


In [ ]:
def moran_i(x, W):
    z = x - x.mean()
    S0 = W.sum()
    return len(x) / S0 * (z @ W @ z) / (z @ z)

I_obs = moran_i(values, W)
E_I = -1 / (len(values) - 1)

permutations = 999
I_perm = np.array([moran_i(rng.permutation(values), W) for _ in range(permutations)])
p_two_sided = (1 + np.sum(np.abs(I_perm - E_I) >= abs(I_obs - E_I))) / (permutations + 1)

print(f"observed I = {I_obs:.3f}")
print(f"E[I] under randomization = {E_I:.3f}")
print(f"two-sided permutation p = {p_two_sided:.4f}")


Moran's \(I\) is **not generally bounded by \([-1,1]\)**. Its attainable range depends on the weight matrix.

A permutation test keeps the locations and weights fixed and asks what \(I\) would look like if the observed values were exchangeable over those locations.

In [ ]:
z = values - values.mean()
lag = W @ z

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(z, lag)
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set(xlabel="centered value z", ylabel="spatial lag Wz", title="Moran scatterplot")
plt.show()


The four quadrants of the Moran scatterplot distinguish high-high, low-low, high-low, and low-high configurations. Local statistics can formalize that idea, but location-by-location inference creates a multiple-testing problem.